# PyTorch Fundus Image Classification — Model Evaluation

This notebook evaluates **3 state-of-the-art PyTorch models** on the fundus image dataset
and exports results as `pytorch_results.json` for the Streamlit app.

**Models:**
- ConvNeXtV2-Tiny (2023)
- SwinV2-Tiny (2022)
- MaxViT-Tiny (2022)

**Run this on Google Colab with GPU enabled.**

In [ ]:
!pip install -q timm torchmetrics

In [ ]:
import os
import json
import time
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
from torchmetrics.classification import (
    MulticlassAccuracy,
    MulticlassF1Score,
    MulticlassAUROC,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

CLASS_NAMES = [
    "AMD", "Bleeding", "Blur Fundus", "Cataract", "Coats", "Cotton Wool Spots",
    "Diabetic Retinopathy", "Drusen", "Glaucoma", "Hemorrhage",
    "Healthy", "Hypertensive Retinopathy", "Laser Scars", "Macular Hole",
    "Maculopathy", "Myopia", "Normal Fundus", "Optic Disc Pallor",
    "Preretinal Hemorrhage", "Proliferative DR", "Retinal Detachment",
    "Retinitis Pigmentosa", "Retinoblastoma", "STARE Normal",
    "Toxoplasmosis", "Vessel Tortuosity",
]

NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4

MODELS = {
    "ConvNeXtV2-Tiny": {"timm_id": "convnextv2_tiny.fcmae_ft_in22k_in1k", "img_size": 224},
    "SwinV2-Tiny":     {"timm_id": "swinv2_tiny_window8_256",             "img_size": 256},
    "MaxViT-Tiny":     {"timm_id": "maxvit_tiny_tf_224.in1k",             "img_size": 224},
}

print(f"Classes: {NUM_CLASSES}")
print(f"Models to evaluate: {list(MODELS.keys())}")

## Upload / Mount Your Dataset

Update `DATASET_ROOT` to point to your fundus image dataset.
Expected structure:
```
dataset/
  train/
    AMD/
      img001.jpg
      ...
    Bleeding/
      ...
  test/
    AMD/
      ...
```

If your dataset is in Google Drive, mount it first.

In [ ]:
# Mount Google Drive (uncomment if your dataset is on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

DATASET_ROOT = "./dataset"  # Update this path
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

assert os.path.isdir(TRAIN_DIR), f"Train directory not found: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Test directory not found: {TEST_DIR}"

In [ ]:
# ── Dataset & DataLoader ───────────────────────────────────────────────────

class FundusDataset(Dataset):
    """Load fundus images from a directory with class-name subfolders."""

    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label_idx, class_name in enumerate(CLASS_NAMES):
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.isdir(class_dir):
                continue
            for fname in os.listdir(class_dir):
                fpath = os.path.join(class_dir, fname)
                if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff")):
                    self.samples.append((fpath, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


def make_transforms(img_size):
    """Create train/test transforms for the given image size."""
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    test_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, test_tf


def make_loaders(img_size):
    """Create train/test DataLoaders for the given image size."""
    train_tf, test_tf = make_transforms(img_size)
    train_ds = FundusDataset(TRAIN_DIR, transform=train_tf)
    test_ds = FundusDataset(TEST_DIR, transform=test_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=2, pin_memory=True)
    print(f"  Train samples: {len(train_ds)}, Test samples: {len(test_ds)}")
    return train_loader, test_loader


print("Dataset & DataLoader ready.")

In [ ]:
# ── Model builder ─────────────────────────────────────────────────────────

def create_model(timm_id: str) -> nn.Module:
    """Create a timm model with a custom classification head."""
    model = timm.create_model(
        timm_id,
        pretrained=True,
        num_classes=NUM_CLASSES,
    )
    return model.to(device)


print("Model builder ready.")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────

def train_model(model, model_name, train_loader, epochs=EPOCHS, lr=LR):
    """Fine-tune a pretrained model on the fundus dataset."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        scheduler.step()
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        print(f"  [{model_name}] Epoch {epoch+1}/{epochs} — "
              f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

    return model


print("Training function ready.")

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_model(model, test_loader):
    """Evaluate model and return accuracy, F1, AUC, and inference time."""
    model.eval()

    acc_metric = MulticlassAccuracy(num_classes=NUM_CLASSES).to(device)
    f1_metric = MulticlassF1Score(num_classes=NUM_CLASSES, average="weighted").to(device)
    auc_metric = MulticlassAUROC(num_classes=NUM_CLASSES, average="macro").to(device)

    total_time = 0.0
    total_samples = 0

    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        start = time.perf_counter()
        outputs = model(images)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        total_time += time.perf_counter() - start

        probs = torch.softmax(outputs, dim=1)
        acc_metric.update(probs, labels)
        f1_metric.update(probs, labels)
        auc_metric.update(probs, labels)
        total_samples += labels.size(0)

    return {
        "accuracy": acc_metric.compute().item(),
        "f1_weighted": f1_metric.compute().item(),
        "auc_macro": auc_metric.compute().item(),
        "inference_ms": (total_time / total_samples) * 1000,
    }


print("Evaluation function ready.")

In [ ]:
# ── Run all models ────────────────────────────────────────────────────────

all_results = {}

for model_name, cfg in MODELS.items():
    timm_id = cfg["timm_id"]
    img_size = cfg["img_size"]

    print(f"\n{'='*60}")
    print(f"Training: {model_name} ({timm_id}) — input {img_size}x{img_size}")
    print(f"{'='*60}")

    train_loader, test_loader = make_loaders(img_size)
    model = create_model(timm_id)
    model = train_model(model, model_name, train_loader)

    print(f"\nEvaluating {model_name}...")
    results = evaluate_model(model, test_loader)
    all_results[model_name] = results

    print(f"  Accuracy:     {results['accuracy']:.4f}")
    print(f"  F1 (weighted):{results['f1_weighted']:.4f}")
    print(f"  AUC (macro):  {results['auc_macro']:.4f}")
    print(f"  Inference:    {results['inference_ms']:.2f} ms/image")

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print("\nAll models evaluated!")

In [ ]:
# ── Export results ────────────────────────────────────────────────────────

output_path = "pytorch_results.json"
with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to {output_path}")
print("\nDownload this file and place it in your intelligenteye/ project root.")
print(json.dumps(all_results, indent=2))

In [ ]:
# ── Download the results file ─────────────────────────────────────────────

try:
    from google.colab import files
    files.download(output_path)
    print("Download triggered!")
except ImportError:
    print("Not running on Colab — copy pytorch_results.json manually.")

## Next Steps

1. Download `pytorch_results.json` from the output above
2. Place it in your `intelligenteye/` project root (next to `app.py`)
3. Run `streamlit run app.py` — the PyTorch results will appear automatically

The Streamlit app displays these benchmark results alongside the live
TensorFlow model inference, giving you the best of both frameworks.